In [0]:
%pip install sentence-transformers faiss-cpu langchain langchain-community langchain-huggingface transformers accelerate

In [0]:
dbutils.library.restartPython()

In [0]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

faiss_index_path = "/Volumes/workspace/365pdf/365pdfupload/faiss_intro_tableau_index"

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.load_local(
    faiss_index_path,
    embedding_model,
    allow_dangerous_deserialization=True
)

print("FAISS vector index loaded successfully")

In [0]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

print("Free FLAN-T5 model loaded successfully")

In [0]:
def build_prompt(question, context):
    prompt = f"""
You are a helpful PDF question-answering assistant.

Answer the question using only the context below.
If the answer is not in the context, say:
"I could not find this information in the uploaded PDF."

Context:
{context}

Question:
{question}

Answer:
"""
    return prompt

In [0]:
def generate_answer_with_flan_t5(prompt, max_new_tokens=200):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [0]:
RELEVANCE_THRESHOLD = 1.3

def ask_pdf_with_llm(question, k=4):
    results_with_scores = vector_store.similarity_search_with_score(
        query=question,
        k=k
    )

    if not results_with_scores:
        return "I could not find any relevant information in the uploaded PDF."

    best_doc, best_score = results_with_scores[0]

    print("Best retrieval score:", best_score)

    if best_score > RELEVANCE_THRESHOLD:
        return "I could not find a confident answer in the uploaded PDF. Please ask a question related to the document."

    context_parts = []
    source_lines = []

    for i, (doc, score) in enumerate(results_with_scores, start=1):
        context_parts.append(doc.page_content)
        source_lines.append(
            f"- {doc.metadata.get('source')}, chunk_id={doc.metadata.get('chunk_id')}, score={score:.4f}"
        )

    context = "\n\n".join(context_parts)
    context = context[:2500]

    prompt = build_prompt(question, context)

    answer = generate_answer_with_flan_t5(prompt)

    final_response = f"""
Answer:
{answer}

Sources:
{chr(10).join(source_lines)}
"""

    return final_response

In [0]:
print(ask_pdf_with_llm("What is Tableau used for?"))

In [0]:
print(ask_pdf_with_llm("What are dimensions and measures in Tableau?"))

In [0]:
print(ask_pdf_with_llm("What is the capital of Japan?"))

In [0]:
print("PDF Q&A Chatbot with Free LLM is ready.")
print("Ask a question about your uploaded PDF.")
print("Type 'exit' to stop.")
print("-" * 80)

while True:
    question = input("You: ")

    if question.lower().strip() in ["exit", "quit", "stop"]:
        print("Chatbot: Goodbye.")
        break

    answer = ask_pdf_with_llm(question)
    print("\nChatbot:")
    print(answer)
    print("-" * 80)